### Notebook *NB04b – Modelos de Machine Learning con FULL sentimiento (horizonte 5 dias)*
**Autor:** Jesús Daniel Romeral Cortina

**Objetivo:**  
Entrenar y evaluar distintos modelos de machine learning utilizando variables financieras del S&P 500 junto con un conjunto ampliado de variables de sentimiento extraídas de noticias financieras (medianas, desviación típica y ratios), con el fin de analizar si la incorporación de sentimiento mejora la capacidad predictiva en la predicción direccional a 5 días frente al baseline sin sentimiento y a los modelos con sentmiento simplificado.

**Metodología (resumen):**
- Se aumentan las features de sentimiento a partir del dataset `sp500_sent_FULL.csv`.
- **Split temporal**: train anterior a **2022-01-01** y test posterior.
- Los hiperparámetros se cargan desde `best_params_ml_5d.json`.
- Métrica principal: **ROC_AUC** (además de Acc, B_Acc, Precision, Recall y F1).
- Métricas guardadas en `resultados_ml_5d_SENT_FULL.csv`.



In [1]:
import pandas as pd
import numpy as np
import json
import os
from sklearn.preprocessing import StandardScaler 
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier 
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix, roc_auc_score, precision_score, recall_score


In [2]:
with open("../../resultados/best_params_ml_5d.json", "r") as f:
    BEST_PARAMS_5D = json.load(f)

In [3]:

FAMILIA = "ML" 
HORIZONTE = "5d" 
SENTIMIENTO = "FULL"

In [4]:
SEED = 42
np.random.seed(SEED)


In [5]:
MODEL_PATH   = "../../datos/sp500_sent_FULL.csv"
OUT_PATH = "../../resultados/resultados_ml_5d_SENT_FULL.csv"

In [6]:
df = pd.read_csv(MODEL_PATH, parse_dates=["Date"])
df = df.sort_values("Date").set_index("Date")

In [7]:
print("Shape:", df.shape)
print("Rango fechas:", df.index.min(), "-", df.index.max())
df.head(3)



Shape: (2811, 28)
Rango fechas: 2013-01-02 00:00:00 - 2024-03-04 00:00:00


,Close,High,Low,Open,Volume,Return,Target_1d,Return_5d_forward,Target_5d,ret_lag_1,...,sentiment_mean,sentiment_median,sentiment_std,n_news,pos_ratio,neg_ratio,neu_ratio,has_news,sentiment_filled,n_news_filled
Date,,,,,,,,,,,,,,,,,,,,,
2013-01-02,1462.420044,1462.430054,1426.189941,1426.189941,4202600000,0.025403,0,-0.000957,0,0.016942,...,0.470913,0.448076,0.545041,4.0,0.5,0.0,0.5,1,0.470913,4
2013-01-03,1459.369995,1465.469971,1455.530029,1462.420044,3829730000,-0.002086,1,0.008737,1,0.025403,...,0.000000,0.000000,0.000000,2.0,0.0,0.0,1.0,1,0.000000,2
2013-01-04,1466.469971,1467.939941,1458.989990,1459.369995,3424290000,0.004865,0,0.003805,1,-0.002086,...,0.000000,0.000000,0.000000,1.0,0.0,0.0,1.0,1,0.000000,1


**Target (1 día)**
- Se incluyen más features de sentimiento como (nombrar columnas))

In [8]:
Y = df["Target_5d"]
X = df.drop(columns=[
    "Target_1d", 
    "Target_5d", 
    "Return_5d_forward",
    "Close",
    "High",
    "Low",
    "Open",
    "Volume",
    "sentiment_mean",
    "n_news"
])
X.head(3)

,Return,ret_lag_1,ret_lag_2,ret_lag_3,ret_lag_4,ret_lag_5,ret_ma_5,ret_std_5,ret_ma_10,ret_std_10,sentiment_median,sentiment_std,pos_ratio,neg_ratio,neu_ratio,has_news,sentiment_filled,n_news_filled
Date,,,,,,,,,,,,,,,,,,
2013-01-02,0.025403,0.016942,-0.011050,-0.001218,-0.004787,-0.002440,0.005058,0.015419,0.002286,0.012203,0.448076,0.545041,0.5,0.0,0.5,1,0.470913,4
2013-01-03,-0.002086,0.025403,0.016942,-0.011050,-0.001218,-0.004787,0.005598,0.015030,0.000928,0.011814,0.000000,0.000000,0.0,0.0,1.0,1,0.000000,2
2013-01-04,0.004865,-0.002086,0.025403,0.016942,-0.011050,-0.001218,0.006815,0.014580,0.002174,0.011468,0.000000,0.000000,0.0,0.0,1.0,1,0.000000,1


**Split temporal por fecha (train/test)**

In [9]:
H = 5
train_mask = df.index < "2022-01-01"

X_train_raw = X.loc[train_mask].iloc[:-H]
y_train     = Y.loc[train_mask].iloc[:-H]

X_test_raw  = X.loc[~train_mask]
y_test      = Y.loc[~train_mask]


In [10]:
def evaluar_modelo(y_test, y_pred, y_proba, nombre):
    metrics = {
        "Familia": FAMILIA,
        "Modelo": nombre,
        "Horizonte": HORIZONTE,
        "Sentimiento" : SENTIMIENTO,
        "Acc": accuracy_score(y_test, y_pred),
        "B_Acc": balanced_accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, y_proba),
        "Conf_Matrix": confusion_matrix(y_test, y_pred),
    }
    return metrics

resultados = []

**Baseline (Dummy)**

In [11]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train_raw, y_train)
y_pred = dummy.predict(X_test_raw)
y_proba = dummy.predict_proba(X_test_raw)[:,1]
resultados.append(evaluar_modelo(y_test, y_pred, y_proba, "Dummy_MostFreq"))

**Logistic Regression**

In [12]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw) 

logreg = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    **BEST_PARAMS_5D["LogisticRegression"]
)

logreg.fit(X_train_scaled, y_train)
y_pred = logreg.predict(X_test_scaled)
y_proba = logreg.predict_proba(X_test_scaled)[:, 1]
resultados.append(evaluar_modelo(y_test, y_pred, y_proba, "Logistic_Reg"))


**Random Forest**

In [13]:
rf = RandomForestClassifier(
    max_features='sqrt',
    random_state=SEED,
    class_weight="balanced",
    n_jobs=-1,
    **BEST_PARAMS_5D["RandomForest"]
)
rf.fit(X_train_raw, y_train)
y_pred = rf.predict(X_test_raw)
y_proba = rf.predict_proba(X_test_raw)[:, 1]
resultados.append(evaluar_modelo(y_test, y_pred, y_proba, "Random_Forest"))

**HistGradientBoosting (Scikit-Learn native)**

In [14]:
hgb = HistGradientBoostingClassifier(
    random_state=SEED,
    **BEST_PARAMS_5D["HistGradientBoosting"]
)
hgb.fit(X_train_raw, y_train)
y_pred = hgb.predict(X_test_raw)
y_proba = hgb.predict_proba(X_test_raw)[:, 1]
resultados.append(evaluar_modelo(y_test, y_pred, y_proba, "Hist_GB"))

**Guardado de resultados**
- Exportación a CSV para combinar en Notebook de resultados `NB05a_Resultados.ipynb`


In [15]:
os.makedirs("../../resultados", exist_ok=True)
df_res = pd.DataFrame(resultados)
df_res["Conf_Matrix"] = df_res["Conf_Matrix"].astype(str).str.replace("\n", " ", regex=False)
df_res.to_csv(OUT_PATH, index=False)
if os.path.exists(OUT_PATH): 
    print(f"Archivo guardado correctamente en {OUT_PATH}") 
else: 
    print("Error: el archivo no se ha guardado.")

print(df_res)


Archivo guardado correctamente en ../../resultados/resultados_ml_5d_SENT_FULL.csv
  Familia          Modelo Horizonte Sentimiento       Acc     B_Acc  \
0      ML  Dummy_MostFreq        5d        FULL  0.556985  0.500000   
1      ML    Logistic_Reg        5d        FULL  0.566176  0.525656   
2      ML   Random_Forest        5d        FULL  0.529412  0.480766   
3      ML         Hist_GB        5d        FULL  0.529412  0.485012   

   Precision    Recall        F1   ROC_AUC             Conf_Matrix  
0   0.556985  1.000000  0.715466  0.500000  [[  0 241]  [  0 303]]  
1   0.571734  0.881188  0.693506  0.557044  [[ 41 200]  [ 36 267]]  
2   0.546720  0.907591  0.682382  0.462923  [[ 13 228]  [ 28 275]]  
3   0.548654  0.874587  0.674300  0.468647  [[ 23 218]  [ 38 265]]  
